# NOAA GHCN-D — Validación local de Parquet (2025)

**Objetivo:** comprobar los dos archivos generados por el ETL, contrastando las agregaciones mensuales contra los registros diarios, sin acceder a AWS ni modificar archivos.

In [2]:
from pathlib import Path
import calendar
import numpy as np
import pandas as pd

In [3]:
ARCHIVO_DIARIO = Path("muestras/diario_datos.parquet")
ARCHIVO_MENSUAL = Path("muestras/mensual_datos.parquet")
ANIO_ESPERADO = 2025

for ruta in (ARCHIVO_DIARIO, ARCHIVO_MENSUAL):
    if not ruta.is_file():
        raise FileNotFoundError(f"No se encontró {ruta.resolve()}")

diario = pd.read_parquet(ARCHIVO_DIARIO)
mensual = pd.read_parquet(ARCHIVO_MENSUAL)

print("DIARIO:", len(diario), "filas")
print(diario.dtypes.to_string())
print("\nMENSUAL:", len(mensual), "filas")
print(mensual.dtypes.to_string())
print("\nEjemplo diario:")
display(diario.head(3))
print("Ejemplo mensual:")
display(mensual.head(3))

DIARIO: 1059 filas
STATION               str
DATE               object
ELEMENT               str
valor_original      int32
valor             float64
M_FLAG                str
S_FLAG                str
OBS_TIME              str

MENSUAL: 8 filas
STATION                  str
mes                    int32
dias_mes               int32
dias_observados        int32
cobertura_pct        float64
prcp_acumulada_mm    float64

Ejemplo diario:


,STATION,DATE,ELEMENT,valor_original,valor,M_FLAG,S_FLAG,OBS_TIME
0,NZ000093012,2025-01-01,TAVG,190,19.0,H,S,
1,NZ000093012,2025-01-02,TAVG,191,19.1,H,S,
2,NZ000093012,2025-01-03,TAVG,181,18.1,H,S,


Ejemplo mensual:


,STATION,mes,dias_mes,dias_observados,cobertura_pct,prcp_acumulada_mm
0,NZ000093292,1,31,31,100.0,82.8
1,NZ000093292,2,28,28,100.0,36.0
2,NZ000093292,3,31,31,100.0,135.8


## Control 1 — Estructura e integridad básica

Se comprueban las columnas necesarias, el año, fechas, claves duplicadas y que los valores diarios de `PRCP` sean numéricos.

In [4]:
def columna(df, nombre):
    coincidencias = [c for c in df.columns if c.lower() == nombre.lower()]
    if len(coincidencias) != 1:
        raise ValueError(f"Se esperaba exactamente una columna {nombre!r}; disponibles: {list(df.columns)}")
    return coincidencias[0]

cd = {x: columna(diario, x) for x in ("STATION", "DATE", "ELEMENT", "valor")}
cm = {x: columna(mensual, x) for x in ("STATION", "mes", "dias_mes", "dias_observados", "cobertura_pct", "prcp_acumulada_mm")}

fecha = pd.to_datetime(diario[cd["DATE"]], errors="coerce")
assert fecha.notna().all(), "Hay fechas diarias inválidas"
assert (fecha.dt.year == ANIO_ESPERADO).all(), "Hay fechas diarias fuera de 2025"

meses = pd.to_numeric(mensual[cm["mes"]], errors="coerce")
assert meses.notna().all() and meses.between(1, 12).all(), "Hay meses mensuales inválidos"

claves_diarias = pd.DataFrame({"estacion": diario[cd["STATION"]], "fecha": fecha, "variable": diario[cd["ELEMENT"]]})
claves_mensuales = pd.DataFrame({"estacion": mensual[cm["STATION"]], "mes": meses.astype(int)})
dup_diarios = int(claves_diarias.duplicated().sum())
dup_mensuales = int(claves_mensuales.duplicated().sum())

prcp = diario[diario[cd["ELEMENT"]].eq("PRCP")].copy()
prcp["_fecha"] = fecha.loc[prcp.index]
prcp["_valor"] = pd.to_numeric(prcp[cd["valor"]], errors="coerce")
prcp_invalidos = int(prcp["_valor"].isna().sum())
prcp_negativos = int((prcp["_valor"] < 0).sum())

print("Estaciones diarias:", diario[cd["STATION"]].nunique())
print("Estaciones mensuales:", mensual[cm["STATION"]].nunique())
print("Duplicados estación-fecha-variable:", dup_diarios)
print("Duplicados estación-mes:", dup_mensuales)
print("PRCP con valor nulo/no numérico:", prcp_invalidos)
print("PRCP negativas:", prcp_negativos)

assert dup_diarios == 0 and dup_mensuales == 0, "Se detectaron claves duplicadas"
assert prcp_invalidos == 0, "Hay PRCP no numéricas"
assert prcp_negativos == 0, "Hay PRCP negativas"
print("Control 1: OK")

Estaciones diarias: 2
Estaciones mensuales: 1
Duplicados estación-fecha-variable: 0
Duplicados estación-mes: 0
PRCP con valor nulo/no numérico: 0
PRCP negativas: 0
Control 1: OK


## Control 2 — Recalcular los agregados mensuales

Se recalculan las sumas y días válidos **directamente desde PRCP diario**, y se comparan con el Parquet mensual por estación y mes. No se rellenan días faltantes ni se exige una cobertura del 100 %. Si un mes no tiene registros PRCP válidos, no se espera una fila mensual.

In [5]:
prcp["_mes"] = prcp["_fecha"].dt.month
calculado = (prcp.groupby([cd["STATION"], "_mes"], as_index=False)
             .agg(prcp_calculada=("_valor", "sum"), dias_calculados=("_fecha", "nunique")))
calculado = calculado.rename(columns={cd["STATION"]: "estacion", "_mes": "mes"})
calculado["dias_mes_calculados"] = calculado["mes"].map(lambda m: calendar.monthrange(ANIO_ESPERADO, int(m))[1])
calculado["cobertura_calculada"] = 100 * calculado["dias_calculados"] / calculado["dias_mes_calculados"]

publicado = mensual.rename(columns={
    cm["STATION"]: "estacion", cm["mes"]: "mes", cm["dias_mes"]: "dias_mes_publicados",
    cm["dias_observados"]: "dias_publicados", cm["cobertura_pct"]: "cobertura_publicada",
    cm["prcp_acumulada_mm"]: "prcp_publicada"
})[["estacion", "mes", "dias_mes_publicados", "dias_publicados", "cobertura_publicada", "prcp_publicada"]].copy()
publicado["mes"] = pd.to_numeric(publicado["mes"]).astype(int)

comparacion = calculado.merge(publicado, on=["estacion", "mes"], how="outer", indicator=True)
comparacion["coincide_filas"] = comparacion["_merge"].eq("both")
comparacion["coincide_dias_mes"] = comparacion["dias_mes_calculados"].eq(comparacion["dias_mes_publicados"])
comparacion["coincide_dias"] = comparacion["dias_calculados"].eq(comparacion["dias_publicados"])
comparacion["coincide_prcp"] = np.isclose(comparacion["prcp_calculada"], comparacion["prcp_publicada"], atol=0.000001, rtol=0, equal_nan=False)
comparacion["coincide_cobertura"] = np.isclose(comparacion["cobertura_calculada"], comparacion["cobertura_publicada"], atol=0.011, rtol=0, equal_nan=False)

controles = ["coincide_filas", "coincide_dias_mes", "coincide_dias", "coincide_prcp", "coincide_cobertura"]
comparacion["ok"] = comparacion[controles].all(axis=1)

print("Meses recalculados:", len(calculado))
print("Filas del mensual:", len(publicado))
print("Meses correctos:", int(comparacion["ok"].sum()))
print("Meses con diferencias:", int((~comparacion["ok"]).sum()))
print("\nDetalle comparado (todas las filas de 2025):")
display(comparacion.sort_values(["estacion", "mes"]).reset_index(drop=True))

if not comparacion["ok"].all():
    print("ATENCIÓN: hay diferencias entre diario y mensual; revisar las filas con ok=False.")
else:
    print("Control 2: OK — acumulado, días y cobertura coinciden en todos los meses presentes")

Meses recalculados: 8
Filas del mensual: 8
Meses correctos: 8
Meses con diferencias: 0

Detalle comparado (todas las filas de 2025):


,estacion,mes,prcp_calculada,dias_calculados,dias_mes_calculados,cobertura_calculada,dias_mes_publicados,dias_publicados,cobertura_publicada,prcp_publicada,_merge,coincide_filas,coincide_dias_mes,coincide_dias,coincide_prcp,coincide_cobertura,ok
0,NZ000093292,1,82.8,31,31,100.000000,31,31,100.000000,82.8,both,True,True,True,True,True,True
1,NZ000093292,2,36.0,28,28,100.000000,28,28,100.000000,36.0,both,True,True,True,True,True,True
2,NZ000093292,3,135.8,31,31,100.000000,31,31,100.000000,135.8,both,True,True,True,True,True,True
3,NZ000093292,4,97.4,30,30,100.000000,30,30,100.000000,97.4,both,True,True,True,True,True,True
4,NZ000093292,5,48.9,29,31,93.548387,31,29,93.548387,48.9,both,True,True,True,True,True,True
5,NZ000093292,6,84.4,30,30,100.000000,30,30,100.000000,84.4,both,True,True,True,True,True,True
6,NZ000093292,7,134.5,30,31,96.774194,31,30,96.774194,134.5,both,True,True,True,True,True,True
7,NZ000093292,8,73.1,24,31,77.419355,31,24,77.419355,73.1,both,True,True,True,True,True,True


Control 2: OK — acumulado, días y cobertura coinciden en todos los meses presentes


## Conclusión

La comparación verifica coherencia **interna** entre los dos Parquet descargados. No demuestra que el CSV RAW original se haya procesado íntegramente ni valida estaciones ausentes de ambos archivos.

In [6]:
print("RESUMEN")
print(f"Diario: {len(diario):,} filas; mensual: {len(mensual):,} filas")
print(f"Meses consistentes: {int(comparacion['ok'].sum())}/{len(comparacion)}")
print("RESULTADO:", "VALIDACIÓN LOCAL SATISFACTORIA" if comparacion["ok"].all() else "REVISAR DIFERENCIAS")

RESUMEN
Diario: 1,059 filas; mensual: 8 filas
Meses consistentes: 8/8
RESULTADO: VALIDACIÓN LOCAL SATISFACTORIA
